# S3 · Ejercicio 2 — Indexar el geocodificador en Elasticsearch

**Curso de Big Data · Dr. Abel Coronado** · Semana 3

> ⚙️ **Laboratorio BDP v7 (Codespaces o portable) · Spark 4.1 · Scala 2.13 · Sedona 1.9.1.**
> Un solo cuaderno para todas las plataformas, con el kernel **BDP v7 (Spark 4.1 · Python 3.12)**. Datos de práctica: recorte de Aguascalientes, cargado en HDFS con `python3 bdp_start.py datos`.

🎯 **Objetivo.** Indexar los domicilios procesados en Elasticsearch (location como geo_point) con el cliente Python (bulk).

📦 **Datos.** `/data/processed/geocoder_final_parquet` (salida del Ejercicio 1).

✅ **Prerrequisitos.** Ejercicio 1 completado. Elasticsearch arrancado (:9200).

📝 **Entregable.** Índice `geocoder_mexico` con docs > 0 (Elasticvue o _count).

⬇️ **Repositorio:** https://github.com/abxda/bdp-lab-nube/tree/main/notebooks/semana_3

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# Celda 1: Configuración de Spark con TODOS los paquetes necesarios

from pyspark.sql import SparkSession

# Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("Configurando una nueva sesión de Spark con capacidades de Sedona y Elasticsearch...")

# Definimos las coordenadas de todos los paquetes que necesitamos en una sola lista
packages = [
    # Paquetes para leer y procesar datos geoespaciales con Sedona
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)

    # Paquete para la escritura nativa en Elasticsearch
]

# Unimos la lista de paquetes en una cadena separada por comas
packages_str = ",".join(packages)

# Configuración y creación de la SparkSession
spark = (
    SparkSession.builder.appName("ProcesoCompleto_Geo_a_ES")
    .master("local[*]")
    .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
    .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

print(f"✅ ¡Sesión de Spark lista con todos los paquetes ({len(packages)}) cargados!")

In [ ]:
from pyspark.sql import functions as sf

# Ruta HDFS al archivo Parquet que ya contiene la dirección completa y las coordenadas
hdfs_path_processed = "hdfs://localhost:9000/data/processed/geocoder_final_parquet"

print(f"Leyendo archivo Parquet desde: {hdfs_path_processed}")
df_direcciones = spark.read.parquet(hdfs_path_processed)

# Transformamos la columna 'location' al formato "lat,lon" que espera el conector nativo
df_para_es = df_direcciones.withColumn(
    "location",
    sf.concat_ws(",", sf.col("Latitud"), sf.col("Longitud"))
).select(
    "DIRECCION_COMPLETA", "CP", "NOM_MUN", "NOM_ENT", "location"
)

print(f"Datos listos para la carga. Total de documentos a indexar: {df_para_es.count():,}")
print("Mostrando una muestra de los datos a enviar a Elasticsearch:")
df_para_es.show(5, truncate=50)

In [ ]:
# Indexación a Elasticsearch con el CLIENTE PYTHON (bulk).
# El conector Spark-ES no existe para Spark 4.x, así que usamos el cliente Python, que es
# robusto y aprovecha que la seguridad de ES está desactivada en el laboratorio.
from elasticsearch import Elasticsearch, helpers

index_name = "geocoder_mexico"
es = Elasticsearch("http://localhost:9200", request_timeout=120)

# Mapeo: location como geo_point; texto y keywords para búsqueda.
mapping = {"mappings": {"properties": {
    "DIRECCION_COMPLETA": {"type": "text"},
    "CP": {"type": "keyword"}, "NOM_MUN": {"type": "keyword"},
    "NOM_ENT": {"type": "keyword"}, "location": {"type": "geo_point"}}}}
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
es.indices.create(index=index_name, body=mapping)

def _gen(rows):
    for r in rows:
        yield {"_index": index_name, "_source": r.asDict()}

# toLocalIterator evita traer todo a memoria del driver de una vez.
total = df_para_es.count()
print(f"Indexando {total:,} documentos en '{index_name}' (cliente Python, bulk)...")
ok, _ = helpers.bulk(es, _gen(df_para_es.toLocalIterator()), chunk_size=2000)
es.indices.refresh(index=index_name)
print(f"\u2705 Indexados {ok:,} documentos. Conteo ES: {es.count(index=index_name)['count']:,}")
